***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 3-Singular value decomposition   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 26, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\horz}{\rule[.5ex]{2.5ex}{0.5pt}}$

## Background: further review of linear algebra

We will need two additional concepts from linear algebra, orthogonal complements and the spectral theorem.

### Orthogonal complement

The *Orthogonal Projection Theorem* implies that any $\mathbf{v} \in \mathbb{R}^n$ can be decomposed into its orthogonal projection onto $U$ and a vector orthogonal to it.

**DEFINITION** **(Orthogonal Complement)** Let $U \subseteq \mathbb{R}^n$ be a linear subspace. The orthogonal complement of $U$, denoted $U^\perp$, is defined as

$$
U^\perp
=
\{\mathbf{w} \in \mathbb{R}^n\,:\, \langle \mathbf{w}, \mathbf{u}\rangle = 0, 
\forall \mathbf{u} \in U\}.
$$

$\natural$

**EXAMPLE:** **(continued)** Continuing the previous example, we compute the orthogonal complement of the linear subspace $W = \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3)$, where $\mathbf{w}_1 = (1,0,1)^T$, $\mathbf{w}_2 = (0,1,1)^T$, and $\mathbf{w}_3 = (1,-1,0)^T$. One way to proceed is to find all vectors that are orthogonal to the orthonormal basis

$$
\mathbf{q}_1
= \frac{1}{\sqrt{2}} \begin{pmatrix}
1\\
0\\
1
\end{pmatrix},
\quad
\mathbf{q}_2
= \frac{1}{\sqrt{6}} \begin{pmatrix}
-1\\
2\\
1
\end{pmatrix}.
$$

We require 

$$
0 = \langle
\mathbf{u}, \mathbf{q}_1
\rangle
= u_1 \left(\frac{1}{\sqrt{2}}\right)
+ u_2 \left(\frac{0}{\sqrt{2}}\right)
+ u_3 \left(\frac{1}{\sqrt{2}}\right)
= \frac{u_1 + u_3}{\sqrt{2}},
$$

$$
0= \langle
\mathbf{u}, \mathbf{q}_2
\rangle
= u_1 \left(-\frac{1}{\sqrt{6}}\right)
+ u_2 \left(\frac{2}{\sqrt{6}}\right)
+ u_3 \left(\frac{1}{\sqrt{6}}\right)
= \frac{-u_1 + 2 u_2 + u_3}{\sqrt{6}}.
$$

The first equation implies $u_3 = -u_1$, which after replacing into the second equation and rearranging gives $u_2 = u_1$.

So all vectors of the form $(u_1,u_1,-u_1)^T$ for some $u_1 \in \mathbb{R}$ are orthogonal to all of $W$. This is a one-dimensional linear subspace. We can choose an orthonormal basis by finding a solution to

$$
1 = (u_1)^2 + (u_1)^2 + (-u_1)^2 = 3 u_1^2, 
$$

Take $u_1 = 1/\sqrt{3}$, that is, let

$$
\mathbf{q}_3
= \frac{1}{\sqrt{3}} \begin{pmatrix}
1\\
1\\
-1
\end{pmatrix}.
$$

Then we have

$$
W^\perp
= \mathrm{span}(\mathbf{q}_3).
$$

$\lhd$

**LEMMA** **(Orthogonal Decomposition)**
Let $U \subseteq \mathbb{R}^n$ be a linear subspace and let $\mathbf{v} \in \mathbb{R}^n$. Then $\mathbf{v}$ can be decomposed as $\mathrm{proj}_U \mathbf{v} + (\mathbf{v} - \mathrm{proj}_U\mathbf{v})$ where $\mathrm{proj}_U \mathbf{v} \in U$ and $(\mathbf{v} - \mathrm{proj}_U \mathbf{v}) \in U^\perp$. Moreover, this decomposition is unique in the following sense: if $\mathbf{v} = \mathbf{u}_1 + \mathbf{u}_2$ with $\mathbf{u}_1 \in U$ and $\mathbf{u}_2 \in U^\perp$, then $\mathbf{u}_1 = \mathrm{proj}_U \mathbf{v}$ and $\mathbf{u}_2 = \mathbf{v} - \mathrm{proj}_U \mathbf{v}$. $\flat$

*Proof:* The first part is an immediate consequence of the *Orthogonal Projection Theorem*. For the second part, assume $\mathbf{v} = \mathbf{u}_1 + \mathbf{u}_2$ with $\mathbf{u}_1 \in U$ and $\mathbf{u}_2 \in U^\perp$. Subtracting $\mathbf{v} = \mathrm{proj}_U \mathbf{v} + (\mathbf{v} - \mathrm{proj}_U\mathbf{v})$, we see that 

$$
(*) \qquad \mathbf{0} = \mathbf{w}_1 + \mathbf{w}_2
$$ 

with

$$
\mathbf{w}_1 = \mathbf{u}_1 - \mathrm{proj}_U \mathbf{v} \in U,
\qquad 
\mathbf{w}_2 = \mathbf{u}_2 - (\mathbf{v} - \mathrm{proj}_U\mathbf{v}) \in U^\perp.
$$

If $\mathbf{w}_1 = \mathbf{w}_2 = \mathbf{0}$, we are done. Otherwise, they must both be nonzero by $(*)$. Further, by the *Properties of Orthonormal Lists*, $\mathbf{w}_1$ and $\mathbf{w}_2$ must be linearly independent. But this is contradicted by the fact that $\mathbf{w}_2 = - \mathbf{w}_1$ by $(*)$. $\square$

**Figure:** Orthogonal decomposition ([Source](https://commons.wikimedia.org/wiki/File:Orthogonal_Decomposition_qtl1.svg))

![Orthogonal decomposition](https://upload.wikimedia.org/wikipedia/commons/thumb/1/12/Orthogonal_Decomposition_qtl1.svg/640px-Orthogonal_Decomposition_qtl1.svg.png)

$\bowtie$

<!--TEX
![Orthogonal decomposition ([Source](https://commons.wikimedia.org/wiki/File:Orthogonal_Decomposition_qtl1.svg))](./figs/2560px-Orthogonal_Decomposition_qtl1.svg.png)
-->

Formally, the *Orthogonal Decomposition Lemma* states that $\mathbb{R}^n$ is a direct sum of any linear subspace $U$ and of its orthogonal complement $U^\perp$: that is, any vector $\mathbf{v} \in \mathbb{R}^n$ can be written uniquely as $\mathbf{v} = \mathbf{u}_1 + \mathbf{u}_2$ with $\mathbf{u}_1 \in U$ and $\mathbf{u}_2 \in U^\perp$. This is denoted $\mathbb{R}^n = U \oplus U^\perp$.

Let $\mathbf{a}_1,\ldots,\mathbf{a}_\ell$ be an orthonormal basis of $U$ and $\mathbf{b}_1,\ldots,\mathbf{b}_k$ be an orthonormal basis of $U^\perp$. By definition of the orthogonal complement, the list

$$
\mathcal{L} = \{\mathbf{a}_1,\ldots,\mathbf{a}_\ell, \mathbf{b}_1,\ldots,\mathbf{b}_k\}
$$

is orthonormal, so it forms a basis of its span. Because any vector in $\mathbb{R}^n$ can be written as a sum of a vector from $U$ and a vector from $U^\perp$, all of $\mathbb{R}^n$ is in the span of $\mathcal{L}$. It follows from the *Dimension Theorem* that $n = \ell + k$, that is,

$$
\mathrm{dim}(U) + \mathrm{dim}(U^\perp) = n.
$$

**EXAMPLE:** **(A Proof of the Rank-Nullity Theorem)** Let $A \in \mathbb{R}^{n \times m}$. Recall that the column space of $A$, $\mathrm{col}(A) \subseteq \mathbb{R}^n$, is the span of all its columns. We compute its othogonal complement. By definition, the columns of $A$, which we denote by $\mathbf{a}_1,\ldots,\mathbf{a}_m$, form a spanning list of $\mathrm{col}(A)$. So $\mathbf{u} \in \mathrm{col}(A)^\perp \subseteq \mathbb{R}^n$ if and only if

$$
\mathbf{a}_i^T\mathbf{u} = \langle \mathbf{u}, \mathbf{a}_i \rangle = 0, \quad \forall i=1,\ldots,m.
$$

Indeed that then implies that for any $\mathbf{v} \in \mathrm{col}(A)$, say $\mathbf{v} = \beta_1 \mathbf{a}_1 + \cdots \beta_m \mathbf{a}_m$ we have

$$
\left\langle \mathbf{u}, \sum_{i=1}^m \beta_i \mathbf{a}_i \right\rangle 
= \sum_{i=1}^m \beta_i \langle \mathbf{u}, \mathbf{a}_i \rangle
= 0.
$$

The $m$ conditions above can be written in matrix form as

$$
A^T \mathbf{u} = \mathbf{0}.
$$

That is, the orthogonal complement of the column space of $A$ is the null space of $A^T$

$$
\mathrm{col}(A)^\perp = \mathrm{null}(A^T).
$$

Applying the same argument to the row space of $A$ which is also the column space of $A^T$, i.e. $\mathrm{row}(A) = \mathrm{col}(A^T) \subseteq \mathbb{R}^m$, it follows that

$$
\mathrm{col}(A^T)^\perp = \mathrm{null}(A),
$$

where note that $\mathrm{null}(A) \subseteq \mathbb{R}^m$. The four linear subspaces $\mathrm{col}(A)$, $\mathrm{col}(A^T)$, $\mathrm{null}(A)$ and $\mathrm{null}(A^T)$ are sometimes referred to as the fundamental subspaces of $A$. We have shown

$$
\mathrm{col}(A) \oplus  \mathrm{null}(A^T) = \mathbb{R}^n
\quad 
\text{and}
\quad
\mathrm{col}(A^T) \oplus  \mathrm{null}(A) = \mathbb{R}^m
$$

By the *Row Rank Equals Column Rank Theorem*, $\mathrm{dim}(\mathrm{col}(A)) = \mathrm{dim}(\mathrm{col}(A^T))$. Moreover, by our previous observation about the dimensions of direct sums, we have

$$
n = \mathrm{dim}(\mathrm{col}(A)) + \mathrm{dim}(\mathrm{null}(A^T))
= \mathrm{dim}(\mathrm{col}(A^T)) + \mathrm{dim}(\mathrm{null}(A^T))
$$

and

$$
m = \mathrm{dim}(\mathrm{col}(A^T)) + \mathrm{dim}(\mathrm{null}(A))
= \mathrm{dim}(\mathrm{col}(A)) + \mathrm{dim}(\mathrm{null}(A)).
$$

So we deduce that

$$
\mathrm{dim}(\mathrm{null}(A))
= m - \mathrm{rk}(A)
$$

and

$$
\mathrm{dim}(\mathrm{null}(A^T))
= n - \mathrm{rk}(A).
$$

These formulas are referred to the *Rank-Nullity Theorem*. The dimension of the null space is sometimes called the nullity. $\lhd$

### Eigenvalues and eigenvectors

Recall the concepts of eigenvalues and eigenvectors. We work on $\mathbb{R}^d$.

**DEFINITION** **(Eigenvalues and Eigenvectors)** Let $A \in \mathbb{R}^{d \times d}$ be a square matrix. Then $\lambda \in \mathbb{R}$ is an eigenvalue of $A$ if there exists a nonzero vector $\mathbf{x} \neq \mathbf{0}$ such that

$$
A \mathbf{x} = \lambda \mathbf{x}.
$$

The vector $\mathbf{x}$ is referred to as an eigenvector. $\natural$

As the next example shows, not every matrix has a (real) eigenvalue.

**EXAMPLE:** **(No Real Eigenvalues)** Set $d = 2$ and let 

$$
A
=
\begin{pmatrix}
0 & -1 \\
1 & 0
\end{pmatrix}.
$$

For $\lambda$ to be an eigenvalue, there must be an nonzero eigenvector $\mathbf{x} = (x_1, x_2)^T$ such that

$$
A \mathbf{x} = \lambda \mathbf{x}
$$

or put differently

$$
- x_2 = \lambda x_1 
\quad\text{and}\quad
x_1 = \lambda x_2.
$$

Replacing these equations into each other, it must be that

$$
- x_2 = \lambda^2 x_2 
\quad\text{and}\quad
x_1 = - \lambda^2 x_1. 
$$

Because $x_1, x_2$ cannot both be $0$, $\lambda$ must satisfy the equation

$$
\lambda^2 = -1
$$

for which there is no real solution. $\lhd$

In general, $A \in \mathbb{R}^{d \times d}$ has at most $d$ distinct eigenvalues.

**LEMMA** **(Number of Eigenvalues)** Let $A \in \mathbb{R}^{d \times d}$ and 
let $\lambda_1, \ldots, \lambda_m$ be distinct eigenvalues of $A$ with corresponding
eigenvectors $\mathbf{x}_1, \ldots, \mathbf{x}_m$. Then $\mathbf{x}_1, \ldots, \mathbf{x}_m$ are linearly independent. In particular, $m \leq d$. $\flat$

**EXAMPLE:** **(Diagonal (and Similar) Matrices)** We use the notation $\mathrm{diag}(\lambda_1,\ldots,\lambda_d)$ for the diagonal matrix with diagonal entries $\lambda_1,\ldots,\lambda_d$. The upper bound in the *Number of Eigenvalues Lemma* can be achieved, for instance, by diagonal matrices with distinct diagonal entries $A = \mathrm{diag}(\lambda_1, \ldots, \lambda_d)$. Each standard basis vector $\mathbf{e}_i$ is then an eigenvector

$$
A \mathbf{e}_i = \lambda_i \mathbf{e}_i.
$$

More generally, let $A$ be similar to a matrix $D = \mathrm{diag}(\lambda_1, \ldots, \lambda_d)$ with distinct diagonal entries, that is, there exists a nonsingular matrix $P$ such that 

$$
A = P D P^{-1}.
$$

Let $\mathbf{p}_1, \ldots, \mathbf{p}_d$ be the columns of $P$. Note that, because the columns of $P$ form a basis of $\mathbb{R}^d$, the entries of the vector $\mathbf{c} = P^{-1} \mathbf{x}$ are the coefficients of the unique linear combination of the $\mathbf{p}_i$'s equal to $\mathbf{x}$. Indeed, $P \mathbf{c} = \mathbf{x}$. Hence, $A \mathbf{x}$ is can be thought of as: (1) expressing $\mathbf{x}$ in the basis $\mathbf{p}_1, \ldots, \mathbf{p}_d$, (2) and scaling the $\mathbf{p}_i$'s by the corresponding $\lambda_i$'s. In particular, the $\mathbf{p}_i$'s are eigenvectors of $A$ since, by the above, $P^{-1} \mathbf{p}_i = \mathbf{e}_i$ and so 

$$
A \mathbf{p}_i 
= P D P^{-1} \mathbf{p}_i 
= P D \mathbf{e}_i
= P (\lambda_i \mathbf{e}_i)
= \lambda_i \mathbf{p}_i.
$$

$\lhd$

**NUMERICAL CORNER:** In Python, the eigenvalues and eigenvectors of a matrix can be computed using [`numpy.linalg.eig`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eig.html).

In [ ]:
A = np.array([[2.5, -0.5], [-0.5, 2.5]])

In [ ]:
w, v = LA.eig(A)
print(w)
print(v)

$\unlhd$

**Spectral theorem** When $A$ is symmetric, that is, $a_{ij} = a_{ji}$ for all $i,j$, a remarkable result is that there exists an orthonormal basis of $\mathbb{R}^d$ made of eigenvectors of $A$. Put differently, $A$ is similar to a diagonal matrix by an orthogonal transformation. We will prove this result below. 

**THEOREM** **(Spectral Theorem)** Let $A \in \mathbb{R}^{d \times d}$ be a symmetric matrix, that is, $A^T = A$. Then $A$ has $d$ orthonormal eigenvectors $\mathbf{q}_1, \ldots, \mathbf{q}_d$ with corresponding (not necessarily distinct) real eigenvalues $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_d$. In matrix form, this is written as the matrix factorization

$$
A 
= Q \Lambda Q^T
= \sum_{i=1}^d \lambda_i \mathbf{q}_i \mathbf{q}_i^T
$$

where $Q$ has columns $\mathbf{q}_1, \ldots, \mathbf{q}_d$ and $\Lambda = \mathrm{diag}(\lambda_1, \ldots, \lambda_d)$. We refer to this factorization as a spectral decomposition of $A$.

$\sharp$

Note that this decomposition indeed produces the eigenvectors of $A$. For any $j$, we have

$$
A \mathbf{q}_j 
= \sum_{i=1}^d \lambda_i \mathbf{q}_i \mathbf{q}_i^T \mathbf{q}_j
= \lambda_j \mathbf{q}_j, 
$$

where we used that, by orthonormality, $\mathbf{q}_i^T \mathbf{q}_j = 0$ if $i \neq j$ and $\mathbf{q}_i^T \mathbf{q}_j = 1$ if $i = j$. The equation above says precisely that $\mathbf{q}_j$ is an eigenvector of $A$ with corresponding eigenvalue $\lambda_j$. Since we have found $d$ eigenvalues (possibly with repetition), we have found all of them.

*Remark:* It can be shown that the sequence of eigenvalues in the *Spectral Theorem* is unique, but that the eigenvectors are not. This will not play a role in our applications. 

**The case of positive semidefinite matrices** The eigenvalues of a symmetric matrix -- while real -- may be negative. There is however an important special case where the eigenvalues are nonnegative, positive semidefinite matrices.

**THEOREM** **(Characterization of Positive Semidefiniteness)** Let $A \in \mathbb{R}^{d \times d}$ be a symmetric matrix and let $A = Q \Lambda Q^T$ be a spectral decomposition of $A$ with $\Lambda = \mathrm{diag}(\lambda_1, \ldots, \lambda_d)$. Then $A \succeq 0$ if and only if its eigenvalues $\lambda_1, \ldots, \lambda_d$ are nonnegative. $\sharp$


*Proof:* Assume $A \succeq 0$. Let $\mathbf{q}_i$ be an eigenvector of $A$ with corresponding eigenvalue $\lambda_i$. Then

$$
\langle \mathbf{q}_i, A \mathbf{q}_i \rangle
= \langle \mathbf{q}_i, \lambda_i \mathbf{q}_i \rangle
= \lambda_i
$$

which must be nonnegative by definition of a positive semidefinite matrix. 

In the other direction, assume $\lambda_1, \ldots, \lambda_d \geq 0$. Note that the vector $Q^T \mathbf{x}$ has entries $\langle \mathbf{q}_i, \mathbf{x}\rangle$, $i=1,\ldots, d$. Thus

$$
\langle \mathbf{x}, A \mathbf{x} \rangle
= \mathbf{x}^T \left(\sum_{i=1}^d \lambda_i \mathbf{q}_i \mathbf{q}_i^T\right) \mathbf{x}
=  \sum_{i=1}^d \lambda_i \mathbf{x}^T\mathbf{q}_i \mathbf{q}_i^T\mathbf{x} 
= \sum_{i=1}^d \lambda_i \langle \mathbf{q}_i, \mathbf{x}\rangle^2
$$

which is necessarily nonnegative. $\square$

**EXAMPLE:** We have seen previously that the Laplacian matrix of a graph $G$ with $n$ vertices is positive semidefinite. Hence it has nonnegative eigenvalues, which are typically denoted

$$
0 \leq \mu_1 \leq \mu_2 \leq \cdots \leq \mu_n.
$$

$\lhd$

**Proof of Spectral Theorem** We start with a proof that distinct eigenvalues correspond to linearly independent eigenvectors.

*Proof:* *(Number of Eigenvalues)* Assume by contradiction that $\mathbf{x}_1, \ldots, \mathbf{x}_m$ are linearly dependent. By the *Linear Dependence Lemma*, there is $k \leq m$ such that

$$
\mathbf{x}_k \in \mathrm{span}(\mathbf{x}_1, \ldots, \mathbf{x}_{k-1})
$$

where $\mathbf{x}_1, \ldots, \mathbf{x}_{k-1}$ are linearly independent. In particular, there are $a_1, \ldots, a_{k-1}$ such that

$$
\mathbf{x}_k = a_1 \mathbf{x}_1 + \cdots + a_{k-1} \mathbf{x}_{k-1}.
$$

Transform the equation above in two ways: (1) multiply both sides by $\lambda_k$ and (2) apply $A$. Then subtract the resulting equations. That leads to

$$
\mathbf{0}
= a_1 (\lambda_k - \lambda_1) \mathbf{x}_1 + \cdots
+ a_{k-1} (\lambda_k - \lambda_{k-1}) \mathbf{x}_{k-1}. 
$$

Because the $\lambda_i$'s are distinct and $\mathbf{x}_1, \ldots, \mathbf{x}_{k-1}$ are linearly independent, we must have $a_1 = \cdots = a_{k-1} = 0$. But that implies that $\mathbf{x}_k = \mathbf{0}$, a contradiction. 

For the second claim, if there were more than $d$ distinct eigenvalues, then there would be more than $d$ corresponding linearly independent eigenvectors by the first claim, a contradiction. $\square$

We will need the following block matrix formula. Suppose 

$$
A
=
\begin{pmatrix}
A_{11} \\ A_{21}
\end{pmatrix}
\quad
\text{and}
\quad
B
=
\begin{pmatrix}
B_{11} & B_{12}
\end{pmatrix}
$$ 

where $A_{11} \in \mathbb{R}^{n_1\times p}$, $A_{21} \in \mathbb{R}^{n_2\times p}$, $B_{11} \in \mathbb{R}^{p\times n_1}$, $B_{12} \in \mathbb{R}^{p\times n_2}$, then

$$
A B
=
\begin{pmatrix}
A_{11} B_{11} & A_{11} B_{12} \\ 
A_{21} B_{11} & A_{21} B_{12}
\end{pmatrix}.
$$

Indeed this is just a special case of the $2 \times 2$ block matrix product formula we have encountered previously, with empty blocks $A_{12}, A_{22}, B_{21}, B_{22}$.

*Proof idea (Spectral Theorem):* Similarly to how we used Householder transformations to "add zeros under the diagonal", here we will use a sequence of orthogonal transformations to add zeros both below and above the diagonal. Specifically, we construct a sequence of orthogonal matrices $\hat{W}_1,\ldots, \hat{W}_d$ such that

$$
\Lambda = W_d^T \cdots W_2^T W_1^T A W_1 W_2 \cdots W_d
$$

is diagonal. Then the matrix $Q$ is simply $W_1 W_2 \cdots W_d$ (check it!). 

To define these matrices, we use a greedy sequence maximizing the [quadratic form](https://en.wikipedia.org/wiki/Quadratic_form#Real_quadratic_forms) $\langle \mathbf{v}, A \mathbf{v}\rangle$. How is this quadratic form related to eigenvalues? Note that, for a unit eigenvector $\mathbf{v}$ with eigenvalue $\lambda$, we have $\langle \mathbf{v}, A \mathbf{v}\rangle = \langle \mathbf{v}, \lambda \mathbf{v}\rangle = \lambda$.

*Proof:* *(Spectral Theorem)* We proceed by induction.

***A first eigenvector:*** Let $A_1 = A$. Define

$$
\mathbf{v}_1 \in \arg\max\{\langle \mathbf{v}, A_1 \mathbf{v}\rangle:\|\mathbf{v}\| = 1\}
$$

which exists by the *Extreme Value Theorem*, and further define

$$
\lambda_1 = \max\{\langle \mathbf{v}, A_1 \mathbf{v}\rangle:\|\mathbf{v}\| = 1\}.
$$

Complete $\mathbf{v}_1$ into an orthonormal basis of $\mathbb{R}^d$, $\mathbf{v}_1$, $\hat{\mathbf{v}}_2, \ldots, \hat{\mathbf{v}}_d$, and form the block matrix

$$
\hat{W}_1 
=
\begin{pmatrix}
\mathbf{v}_1 & \hat{V}_1
\end{pmatrix}
$$

where the columns of $\hat{V}_1$ are $\hat{\mathbf{v}}_2, \ldots, \hat{\mathbf{v}}_d$. Note that $\hat{W}_1$ is orthogonal by construction.

***Getting one step closer to diagonalization:*** We show next that $W_1$ gets us one step closer to a diagonal matrix by similarity transformation. Note first that

\begin{align*}
\hat{W}_1^T A_1 \hat{W}_1
&=
\begin{pmatrix}
\mathbf{v}_1^T \\ \hat{V}_1^T
\end{pmatrix}
A_1
\begin{pmatrix}
\mathbf{v}_1 & \hat{V}_1
\end{pmatrix}\\
&=
\begin{pmatrix}
\mathbf{v}_1^T \\ \hat{V}_1^T
\end{pmatrix}
\begin{pmatrix}
A_1 \mathbf{v}_1 & A_1 \hat{V}_1
\end{pmatrix}\\
&=
\begin{pmatrix}
\mathbf{v}_1^T A_1 \mathbf{v}_1 & \mathbf{v}_1^T A_1 \hat{V}_1\\
\hat{V}_1^T A_1 \mathbf{v}_1 & \hat{V}_1^T A_1 \hat{V}_1
\end{pmatrix}\\
&= 
\begin{pmatrix}
\lambda_1 & \mathbf{w}_1^T \\
\mathbf{w}_1 & A_2
\end{pmatrix}
\end{align*}

where $\mathbf{w}_1 = \hat{V}_1^T A_1 \mathbf{v}_1$ and $A_2 = \hat{V}_1^T A_1 \hat{V}_1$.

The key claim is that $\mathbf{w}_1 = \mathbf{0}$. This follows from a contradiction argument. Indeed, suppose $\mathbf{w}_1 \neq \mathbf{0}$ and consider the unit vector (*Exercise 3.65* asks for a proof that $\|\mathbf{z}\| = 1$)

$$
\mathbf{z}
=
\hat{W}_1 
\frac{1}{\sqrt{1 + \delta^2 \|\mathbf{w}_1\|^2}} \begin{pmatrix}
1 \\
\delta \mathbf{w}_1
\end{pmatrix}
$$

which, by the formula in *Exercise 2.52*, achieves the objective value

\begin{align*}
\mathbf{z}^T A_1 \mathbf{z}
&=
\frac{1}{1 + \delta^2 \|\mathbf{w}_1\|^2}
\begin{pmatrix}
1 \\
\delta \mathbf{w}_1
\end{pmatrix}^T
\begin{pmatrix}
\lambda_1 & \mathbf{w}_1^T \\
\mathbf{w}_1 & A_2
\end{pmatrix}
\begin{pmatrix}
1 \\
\delta \mathbf{w}_1
\end{pmatrix}\\
&= 
\frac{1}{1 + \delta^2 \|\mathbf{w}_1\|^2}
\left(
\lambda_1
+ 2 \delta \|\mathbf{w}_1\|^2
+ \delta^2 \mathbf{w}_1^T A_2 \mathbf{w}_1
\right).
\end{align*}

By the [sum of a geometric series](https://en.wikipedia.org/wiki/Geometric_series), for $\varepsilon \in (0,1)$,

$$
\frac{1}{1 + \varepsilon^2}
= 1 - \varepsilon^2 + \varepsilon^4 + \cdots.
$$

So, for $\delta > 0$ small enough,

\begin{align*}
\mathbf{z}^T A_1 \mathbf{z}
&\approx
(\lambda_1
+ 2 \delta \|\mathbf{w}_1\|^2
+ \delta^2 \mathbf{w}_1^T A_2 \mathbf{w}_1)
(1 - \delta^2 \|\mathbf{w}_1\|^2)\\
&\approx \lambda_1 
+ 2 \delta \|\mathbf{w}_1\|^2
+ C \delta^2\\ 
&> \lambda_1
\end{align*}

where $C \in \mathbb{R}$ depends on $\mathbf{w}_1$ and $A_2$, and where we ignored the "higher-order" terms involving $\delta^3, \delta^4, \delta^5, \ldots$ which are negligible. That gives the desired contradiction - that is, it would imply the existence of a vector achieving a stricter better objective value than the optimal $\mathbf{v}_1$. (See *Exercise 3.65* for a more formal derivation.)

So, letting $W_1 = \hat{W}_1$,

$$
W_1^T A_1 W_1
= 
\begin{pmatrix}
\lambda_1 & \mathbf{0} \\
\mathbf{0} & A_2
\end{pmatrix}.
$$

Finally note that $A_2 = \hat{V}_1^T A_1 \hat{V}_1$ is symmetric

$$
A_2^T = (\hat{V}_1^T A_1 \hat{V}_1)^T = \hat{V}_1^T A_1^T \hat{V}_1 = \hat{V}_1^T A_1 \hat{V}_1 = A_2
$$

by the symmetry of $A_1$ itself.

***Next step of the induction:*** Apply the same argument to the symmetric matrix $A_2 \in \mathbb{R}^{(d-1)\times (d-1)}$, let $\hat{W}_2 = (\mathbf{v}_2\ \hat{V}_2) \in \mathbb{R}^{(d-1)\times (d-1)}$ be the corresponding orthogonal matrix, and define $\lambda_2$ and $A_3$ through the equation

$$
\hat{W}_2^T A_2 \hat{W}_2
= 
\begin{pmatrix}
\lambda_2 & \mathbf{0} \\
\mathbf{0} & A_3
\end{pmatrix}.
$$

Now define the block matrix

$$
W_2
=
\begin{pmatrix}
1 & \mathbf{0}\\
\mathbf{0} & \hat{W}_2
\end{pmatrix}.
$$

Observe that (check it!)

$$
W_2^T W_1^T A_1 W_1 W_2
= W_2^T \begin{pmatrix}
\lambda_1 & \mathbf{0} \\
\mathbf{0} & A_2
\end{pmatrix}
W_2
=
\begin{pmatrix}
\lambda_1 & \mathbf{0}\\
\mathbf{0} & \hat{W}_2^T A_2 \hat{W}_2
\end{pmatrix}
=\begin{pmatrix}
\lambda_1 & 0 & \mathbf{0} \\
0 & \lambda_2 & \mathbf{0} \\
\mathbf{0} & \mathbf{0} & A_3
\end{pmatrix}.
$$

Proceeding similarly by induction gives the claim. $\square$

*Remarks:* A couple remarks about the proof:

1. The fact that $\mathbf{w}_1 \neq \mathbf{0}$ is perhaps more intuitively understood through vector calculus by using the [method of Lagrangian multipliers](https://en.wikipedia.org/wiki/Lagrange_multiplier) on $\max\{\langle \mathbf{v}, A_1 \mathbf{v}\rangle:\|\mathbf{v}\| = 1\}$ to see that $A_1 \mathbf{v}_1$ must be proportional to $\mathbf{v}_1$. Hence $\mathbf{w}_1 = \hat{V}_1^T A_1 \mathbf{v}_1 = \mathbf{0}$ by construction of $\hat{V}_1^T$. Making this argument rigorous is beyond this course.

2. By construction, the vector $\mathbf{v}_2$ (i.e., the first column of $\hat{W}_2$) is the solution to

$$
\mathbf{v}_2 \in \arg\max\{\langle \mathbf{v}, A_2 \mathbf{v}\rangle:\|\mathbf{v}\| = 1\}.
$$

Note that, by definition of $A_2$ (and the fact that $A_1 = A$),

$$
\mathbf{v}^T A_2 \mathbf{v}
= \mathbf{v}^T \hat{V}_1^T A_1 \hat{V}_1 \mathbf{v}
= (\hat{V}_1 \mathbf{v})^T \,A \,(\hat{V}_1 \mathbf{v}).
$$

So we can think of the solution $\mathbf{v}_2$ as specifying an optimal linear combination of the columns of $\hat{V}_1$ -- which form a basis of the space $\mathrm{span}(\mathbf{v}_1)^\perp$ of vectors orthogonal to $\mathbf{v}_1$. In essence $\mathbf{v}_2$ solves the same problem as $\mathbf{v}_1$, *but restricted to $\mathrm{span}(\mathbf{v}_1)^\perp$*. We will come back to this below.